# 01 Parse XML and EDA

Parse OhioT1DM-style XML files, clean glucose readings, inspect patient coverage, and save the cleaned dataset.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from src.preprocessing import clean_glucose_dataframe, parse_ohio_xml_folder

RAW_DIR = ROOT / "data" / "raw" / "ohio"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_df = parse_ohio_xml_folder(RAW_DIR)
print(f"Rows parsed: {len(raw_df):,}")
print(f"Patients: {raw_df['patient_id'].nunique() if not raw_df.empty else 0}")
raw_df.head()

In [ ]:
if raw_df.empty:
    raise FileNotFoundError(f"No XML files found under {RAW_DIR}. Add data before continuing.")

clean_df = clean_glucose_dataframe(raw_df)
summary = clean_df.groupby("patient_id").agg(
    rows=("glucose_mgdl", "size"),
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    mean_mgdl=("glucose_mgdl", "mean"),
)
summary

In [ ]:
sample_patients = clean_df["patient_id"].drop_duplicates().head(3)
fig, ax = plt.subplots(figsize=(14, 5))
for patient_id in sample_patients:
    patient_df = clean_df[clean_df["patient_id"] == patient_id].head(500)
    ax.plot(patient_df["timestamp"], patient_df["glucose_mgdl"], label=f"Patient {patient_id}")
ax.axhline(70, color="red", linestyle="--", linewidth=1, label="Low threshold")
ax.axhline(180, color="orange", linestyle="--", linewidth=1, label="High threshold")
ax.set_title("Sample glucose timelines")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Glucose mg/dL")
ax.legend()
plt.tight_layout()

In [ ]:
output_path = PROCESSED_DIR / "ohio_all_patients_clean.csv"
clean_df.to_csv(output_path, index=False)
output_path